# 01 — ESS Load (R6–R11)

**Day 1, Track B.** Build the long-format ESS panel for the analysis.

## What this notebook does

1. Inspects the ESS files on disk (course teaching subset + any integrated SAVs in `data/raw/ess/`).
2. Loads R6–R11 via `src.mla.ess_io.load_full_panel`, gracefully skipping rounds whose SAVs are not yet present.
3. Reports the per-round country × N coverage so we can see exactly where the panel gaps are.
4. Persists the resulting frame as `data/interim/ess_full.parquet` for downstream notebooks.

## Important context

On disk inspection (2026-04-25), the course's `ESSrounds1to9.dta` file is a **narrow teaching subset** (~11 columns: `Country, essround, ais, res_mig, res_educ, lrscale, gndr, agea, gdppc_inter, unemp_rate_inter, infl_for_perc_inter`). It does **not** contain `idno`, weights, trust outcomes, `isco08`, `eisced`, `hinctnta`, `mnactic`, or `domicil`. The integrated SAVs from <https://ess.sikt.no/> are therefore the primary source for **all** rounds R6–R11, not just R10/R11 as the original plan assumed.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "MLA" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.mla.ess_io import (  # noqa: E402
    CORE_COLUMNS,
    RAW_ESS_DIR,
    find_round_file,
    load_full_panel,
    load_teaching_panel,
)

INTERIM_DIR = REPO_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT, RAW_ESS_DIR.exists()

(PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA'), True)

## 1. Inspect what's on disk

In [2]:
presence = pd.DataFrame(
    {
        "round": list(range(6, 12)),
        "sav_or_dta_in_data_raw_ess": [
            find_round_file(r) for r in range(6, 12)
        ],
    }
)
presence["present"] = presence["sav_or_dta_in_data_raw_ess"].notna()
presence

,round,sav_or_dta_in_data_raw_ess,present
0,6,None,False
1,7,None,False
2,8,None,False
3,9,None,False
4,10,None,False
5,11,None,False


## 2. Load whatever is available

In [3]:
# strict=False so the notebook runs even when not all SAVs are dropped yet.
# Day 1 hard checkpoint requires R6–R11 all loaded; treat this as the
# best-effort run while data acquisition is in progress.
try:
    panel = load_full_panel(
        rounds=(6, 7, 8, 9, 10, 11),
        columns=CORE_COLUMNS,
        strict=False,
    )
    print(f"Loaded shape: {panel.shape}")
    print(f"Rounds present: {sorted(panel['essround'].unique().tolist())}")
    have_real_data = True
except RuntimeError as e:
    print("No integrated SAVs available yet:")
    print("  ", str(e).splitlines()[0])
    have_real_data = False

[ess_io] Missing rounds: [6, 7, 8, 9, 10, 11]. Drop ESS{N}.sav into /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/raw/ess or set try_api=True with ESS_USER_ID in .env.
No integrated SAVs available yet:
   No ESS rounds could be loaded. Place integrated SAV files in /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/raw/ess (e.g. ESS6e02_6.sav, …, ESS11e04_0.sav) or supply ESS_USER_ID in .env and pass try_api=True.


## 3. Per-round country × N coverage table

When the integrated panel is in, this drives the Day-1 hard checkpoint check (variable parity vs. R1–R9).

In [4]:
if have_real_data:
    cov = (
        panel.groupby("essround")
        .agg(
            n_individuals=("essround", "size"),
            n_countries=("cntry", "nunique"),
        )
        .sort_index()
    )
    print("Per-round coverage (integrated ESS files):")
    display(cov)
else:
    print("Cannot compute integrated coverage — no integrated SAVs available.")

Cannot compute integrated coverage — no integrated SAVs available.


## 4. Sanity-check coverage against the course's teaching subset

The course teaching subset of R6–R9 is on disk and gives us a baseline expectation for country coverage. The integrated SAVs (once available) should match closely.

In [5]:
teaching = load_teaching_panel(rounds=(6, 7, 8, 9))
teaching_cov = (
    teaching.groupby("essround")
    .agg(
        n_individuals=("essround", "size"),
        n_countries=("Country", "nunique"),
    )
    .sort_index()
)
print("Per-round coverage (course teaching subset, R6–R9 baseline):")
teaching_cov

Per-round coverage (course teaching subset, R6–R9 baseline):


,n_individuals,n_countries
essround,,
6,10157,21
7,9411,19
8,9873,20
9,9354,21


## 5. Persist the integrated panel as `ess_full.parquet`

In [6]:
if have_real_data:
    out_path = INTERIM_DIR / "ess_full.parquet"
    panel.to_parquet(out_path, index=False)
    print(f"Wrote {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")
else:
    print(
        "Skipping parquet write — no integrated panel yet. Drop SAVs into\n"
        f"  {RAW_ESS_DIR}\n"
        "and re-run."
    )

Skipping parquet write — no integrated panel yet. Drop SAVs into
  /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/raw/ess
and re-run.


## 6. Day-1 hard checkpoint

From `IMPLEMENTATION_PLAN.md` §11: *Day 1: ESS R10+R11 loaded with variable parity vs. R1–R9; initial commit pushed.*

Given the discovery that the course's R1–R9 file is a narrow teaching subset, the checkpoint widens to **all of R6–R11 must be loaded from integrated SAVs** (or the API), with all `CORE_COLUMNS` present.

In [7]:
expected_cols = set(CORE_COLUMNS)
if have_real_data:
    have_cols = set(panel.columns)
    missing = expected_cols - have_cols
    extra = have_cols - expected_cols
    rounds_loaded = set(panel["essround"].unique().tolist())
    print(f"Rounds loaded: {sorted(rounds_loaded)}")
    print(f"Missing core columns: {sorted(missing) if missing else 'none'}")
    print(f"Extra columns: {sorted(extra) if extra else 'none'}")
    checkpoint_ok = (
        rounds_loaded == {6, 7, 8, 9, 10, 11} and not missing
    )
    print(f"Day-1 checkpoint: {'PASS' if checkpoint_ok else 'pending'}")
else:
    print("Day-1 checkpoint: blocked — no integrated SAVs available.")

Day-1 checkpoint: blocked — no integrated SAVs available.
